In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = "/content/drive/MyDrive/ntu60_stgcn"
CKPT_DIR = os.path.join(BASE_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import shutil
import time

LOCAL_DIR = "/content/ntu60_data"
os.makedirs(LOCAL_DIR, exist_ok=True)

files = {
    "train_data.npy":   os.path.join(BASE_DIR, "data/train_data.npy"),
    "train_labels.npy": os.path.join(BASE_DIR, "data/train_labels.npy"),
    "test_data.npy":    os.path.join(BASE_DIR, "data/test_data.npy"),
    "test_labels.npy":  os.path.join(BASE_DIR, "data/test_labels.npy"),
}

for fname, src in files.items():
    dst = os.path.join(LOCAL_DIR, fname)
    t   = time.time()
    shutil.copy2(src, dst)
    size_mb = os.path.getsize(dst) / 1e6

print(f"All files copied to {LOCAL_DIR}")


All files copied to /content/ntu60_data


In [9]:
import numpy as np

train_data = np.load(f"{LOCAL_DIR}/train_data.npy")
train_labels = np.load(f"{LOCAL_DIR}/train_labels.npy")
test_data = np.load(f"{LOCAL_DIR}/test_data.npy")
test_labels = np.load(f"{LOCAL_DIR}/test_labels.npy")

print(f"  train_data   : {train_data.shape}  {train_data.dtype}")
print(f"  train_labels : {train_labels.shape}")
print(f"  test_data    : {test_data.shape}")
print(f"  test_labels  : {test_labels.shape}")

  train_data   : (40086, 3, 100, 25)  float32
  train_labels : (40086,)
  test_data    : (16483, 3, 100, 25)
  test_labels  : (16483,)


In [10]:
import random
import numpy as np
import torch
import torch.nn as nn
import os
import time
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Seed fixed to {SEED}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Seed fixed to 42
GPU: NVIDIA A100-SXM4-40GB


In [11]:
def augment_skeleton(joints):
    """
    Enhanced augmentation for skeleton sequences.
    joints: tensor shape (3, 100, 25)

    Applies:
    1. Random rotation around Y axis (+/-15 degrees)
    2. Random scaling (+/-10%)
    3. Random horizontal flip (50% chance)
    4. Random temporal cropping (50% chance)
    5. Random joint dropout (30% chance)
    """
    # 1. Random rotation around Y axis
    angle = np.random.uniform(-15, 15) * np.pi / 180
    cos_a = np.cos(angle)
    sin_a = np.sin(angle)
    rotation = torch.tensor([
        [ cos_a, 0, sin_a],
        [     0, 1,     0],
        [-sin_a, 0, cos_a],
    ], dtype=torch.float32)
    joints = torch.einsum('rc,ctv->rtv', rotation, joints)

    # 2. Random scaling
    scale = np.random.uniform(0.9, 1.1)
    joints = joints * scale

    # 3. Random horizontal flip
    if np.random.random() < 0.5:
        joints = joints.clone()
        joints[0] = -joints[0]

    # 4. Random temporal cropping
    if np.random.random() < 0.5:
        crop_len = int(100 * np.random.uniform(0.85, 1.0))
        start = np.random.randint(0, 100 - crop_len)
        cropped = joints[:, start:start + crop_len, :]
        pad_size = 100 - crop_len
        padding = cropped[:, -1:, :].expand(-1, pad_size, -1)
        joints = torch.cat([cropped, padding], dim=1)

    # 5. Random joint dropout
    if np.random.random() < 0.3:
        mask = (torch.rand(25) > 0.1).float()
        joints = joints * mask.unsqueeze(0).unsqueeze(0)

    return joints


class NTUDataset(Dataset):
    def __init__(self, data, labels, augment=False):
        self.data = data.astype(np.float32)
        self.labels = labels.astype(np.int64)
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.data[idx].copy())
        y = torch.tensor(self.labels[idx])
        if self.augment:
            x = augment_skeleton(x)
        return x, y


BATCH_SIZE = 32

train_dataset = NTUDataset(train_data, train_labels, augment=True)
test_dataset  = NTUDataset(test_data,  test_labels,  augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = 4,
    pin_memory = True,
    prefetch_factor = 4,
)

test_loader = DataLoader(
    test_dataset,
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = 4,
    pin_memory = True,
    prefetch_factor = 4,
)

# Speed test - For Colab check
t = time.time()
b, _ = next(iter(train_loader))
print(f"Batch load time : {time.time()-t:.2f}s")
print(f"Batch shape     : {b.shape}")
print(f"  Dataset ready")
print(f"  Train : {len(train_dataset)} samples (augmentation ON)")
print(f"  Test  : {len(test_dataset)} samples  (augmentation OFF)")
print(f"  Batch size : {BATCH_SIZE}")

Batch load time : 0.16s
Batch shape     : torch.Size([32, 3, 100, 25])
  Dataset ready
  Train : 40086 samples (augmentation ON)
  Test  : 16483 samples  (augmentation OFF)
  Batch size : 32


In [12]:
def build_adjacency_matrix():
  # Define body links
    edges = [
        (0,1),(1,20),(20,2),(2,3),
        (20,4),(4,5),(5,6),(6,7),(7,21),(7,22),
        (20,8),(8,9),(9,10),(10,11),(11,23),(11,24),
        (0,12),(12,13),(13,14),(14,15),
        (0,16),(16,17),(17,18),(18,19),
    ]
    A = np.zeros((25, 25), dtype=np.float32)
    for i in range(25):
        A[i, i] = 1
    for i, j in edges:
        A[i, j] = 1
        A[j, i] = 1

 # Normalize adjacency matrix
    D_inv = np.diag(1.0 / np.sqrt(A.sum(axis=1)))
    A_norm = D_inv @ A @ D_inv
    return torch.FloatTensor(A_norm)

A = build_adjacency_matrix()
print(f"Adjacency matrix : {A.shape}")
print(f"Non-zero entries : {(A > 0).sum().item()}")
print(f"Adjacency matrix ready")

Adjacency matrix : torch.Size([25, 25])
Non-zero entries : 73
Adjacency matrix ready


In [13]:
class GraphConv(nn.Module):
    """
    Spatial Graph Convolution Layer.
    Aggregates features from neighboring joints using the
    normalized adjacency matrix, then applies a learned transformation.

    Input  : (batch, in_channels,  T, 25)
    Output : (batch, out_channels, T, 25)
    """
    def __init__(self, in_channels, out_channels, A):
        super().__init__()
        # Register the normalized adjacency matrix as a buffer
        self.register_buffer("A", A)

        # Learned linear transform per joint per frame
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

        # Normalize output features for training stability
        self.bn   = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        # Apply learned transformation to each joint independently
        x = self.conv(x)

        # Aggregate neighbor joint features using the body graph
        # each joint collects weighted information from its neighbors
        x = torch.einsum("bctv,vw->bctw", x, self.A)

        # Stabilize features across the batch
        x = self.bn(x)
        return x


class STGCNBlock(nn.Module):
    """
    Spatial-Temporal Graph Convolution Block.
    Combines spatial graph convolution (across joints) with
    temporal convolution (across frames) plus a residual connection.
    """
    def __init__(self, in_channels, out_channels, A,
                 stride=1, dropout=0.5):
        super().__init__()
        # Aggregate information from neighboring joints
        self.gcn = GraphConv(in_channels, out_channels, A)

        # Capture motion patterns across 9 consecutive frames
        # kernel (9,1) = looks at 4 frames before and 4 after each frame
        # padding (4,0) = keeps the time dimension unchanged
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(
                out_channels, out_channels,
                kernel_size=(9, 1),
                stride=(stride, 1),
                padding=(4, 0),
            ),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(dropout),
        )

        # Residual connection - adds input back to output
        # prevents gradient vanishing in deep networks
        if in_channels == out_channels and stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels,
                          kernel_size=(1, 1),
                          stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )
        self.relu = nn.ReLU()

    def forward(self, x):
        res = self.residual(x)
        x = self.gcn(x)
        x = self.tcn(x)
        return self.relu(x + res)


class STGCN(nn.Module):
    """
    Processes skeleton sequences through 9 ST-GCN blocks with
    progressively increasing channel sizes, capturing increasingly
    complex spatial-temporal action patterns.

    Input  : (batch, 3, 100, 25)  - 3 coords, 100 frames, 25 joints
    Output : (batch, 60)          - score for each action class
    """
    def __init__(self, num_classes=60, dropout=0.5):
        super().__init__()
        A = build_adjacency_matrix()
        self.register_buffer("A", A)
        self.input_bn = nn.BatchNorm1d(3 * 25)

        # 9 ST-GCN blocks with increasing channel capacity
        # early blocks (64ch)  : individual joint movements
        # middle blocks (128ch): coordinated patterns across multiple joints
        # late blocks (256ch)  : full body action signatures
        self.blocks   = nn.ModuleList([
            STGCNBlock(3,   64,  A, dropout=dropout),
            STGCNBlock(64,  64,  A, dropout=dropout),
            STGCNBlock(64,  64,  A, dropout=dropout),
            STGCNBlock(64,  128, A, dropout=dropout),
            STGCNBlock(128, 128, A, dropout=dropout),
            STGCNBlock(128, 128, A, dropout=dropout),
            STGCNBlock(128, 256, A, dropout=dropout),
            STGCNBlock(256, 256, A, dropout=dropout),
            STGCNBlock(256, 256, A, dropout=dropout),
        ])
        self.dropout    = nn.Dropout(dropout)

        # Final linear layer maps 256 features to 60 class scores
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x = x.permute(0, 1, 3, 2)
        x = x.reshape(B, -1, 100)
        x = self.input_bn(x)
        x = x.reshape(B, 3, 25, 100)
        x = x.permute(0, 1, 3, 2)
        for block in self.blocks:
            x = block(x)

        # Global average pooling - collapse time and joint dimensions
        x = x.mean(dim=[2, 3])
        x = self.dropout(x)
        # Map to 60 class scores - highest score = predicted action
        return self.classifier(x)


# Instantiate
model_stgcn = STGCN(num_classes=60, dropout=0.5).to(device)
total = sum(p.numel() for p in model_stgcn.parameters()
            if p.requires_grad)

with torch.no_grad():
    x = torch.randn(4, 3, 100, 25).to(device)
    out = model_stgcn(x)

In [14]:
# Label smoothing loss
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Adam optimizer
optimizer_stgcn = torch.optim.Adam(
    model_stgcn.parameters(),
    lr = 1e-3,
    weight_decay = 1e-4,
    betas = (0.9, 0.999),
)

# Cosine annealing with warm restarts
scheduler_stgcn = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_stgcn,
    T_0 = 20,     # restart every 20 epochs
    T_mult  = 1,      # keep same period
    eta_min = 1e-5,   # minimum LR
)

best_acc_stgcn = 0.0
history_stgcn  = {
    "train_loss": [], "train_acc": [],
    "test_loss":  [], "test_acc":  [],
    "lr":         [],
}

def save_checkpoint(model, optimizer, epoch, val_acc, filepath):
    torch.save({
        "epoch"      : epoch,
        "model_state": model.state_dict(),
        "optim_state": optimizer.state_dict(),
        "val_acc"    : val_acc,
    }, filepath)

print(f"Loss       : CrossEntropyLoss (label_smoothing=0.1)")
print(f"Optimizer  : Adam (lr=1e-3, weight_decay=1e-4)")
print(f"Scheduler  : CosineAnnealingWarmRestarts (T_0=20, eta_min=1e-5)")
print(f"Batch size : {BATCH_SIZE}")
print(f"Augment    : rotation + scale + flip + temporal crop + joint dropout")
print(f"Seed       : {SEED}")
print(f"\nLR schedule preview:")
lrs = []
opt_tmp = torch.optim.Adam([torch.zeros(1)], lr=1e-3)
sch_tmp = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    opt_tmp, T_0=20, T_mult=1, eta_min=1e-5)
for e in range(60):
    lrs.append(sch_tmp.get_last_lr()[0])
    sch_tmp.step()
for e in [0, 9, 19, 20, 29, 39, 40, 49, 59]:
    print(f"  Epoch {e+1:>2} : {lrs[e]:.6f}")
print(f"Training setup ready")

Loss       : CrossEntropyLoss (label_smoothing=0.1)
Optimizer  : Adam (lr=1e-3, weight_decay=1e-4)
Scheduler  : CosineAnnealingWarmRestarts (T_0=20, eta_min=1e-5)
Batch size : 32
Augment    : rotation + scale + flip + temporal crop + joint dropout
Seed       : 42

LR schedule preview:
  Epoch  1 : 0.001000
  Epoch 10 : 0.000582
  Epoch 20 : 0.000016
  Epoch 21 : 0.001000
  Epoch 30 : 0.000582
  Epoch 40 : 0.000016
  Epoch 41 : 0.001000
  Epoch 50 : 0.000582
  Epoch 60 : 0.000016
Training setup ready


In [9]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = False

NUM_EPOCHS = 60

print(f"Training ST-GCN for {NUM_EPOCHS} epochs")
print(f"\n{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} "
      f"{'Test Loss':>10} {'Test Acc':>10} {'LR':>10} {'Time':>6}")
print("─" * 76)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # Training
    model_stgcn.train()
    train_loss = torch.tensor(0.0, device=device)
    train_correct = torch.tensor(0,   device=device)
    train_total = 0
    train_batches = 0

    loop = tqdm(train_loader,
                desc=f"Epoch {epoch:>2}/{NUM_EPOCHS} [Train]",
                leave=False, ncols=80)

    for data, labels in loop:
        data, labels = data.to(device), labels.to(device)
        optimizer_stgcn.zero_grad()
        outputs = model_stgcn(data)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_stgcn.step()

        train_loss += loss.detach()
        train_correct += (outputs.detach().argmax(1) == labels).sum()
        train_total += len(labels)
        train_batches += 1

        loop.set_postfix(
            loss=f"{loss.item():.4f}",
            acc=f"{100*train_correct.item()/train_total:.1f}%",
        )

    train_loss = train_loss.item() / train_batches
    train_acc  = train_correct.item() / train_total * 100

    # Evaluation
    model_stgcn.eval()
    test_loss = torch.tensor(0.0, device=device)
    test_correct = torch.tensor(0,   device=device)
    test_total = 0
    test_batches = 0

    eval_loop = tqdm(test_loader,
                     desc=f"Epoch {epoch:>2}/{NUM_EPOCHS} [Eval] ",
                     leave=False, ncols=80)

    with torch.no_grad():
        for data, labels in eval_loop:
            data, labels = data.to(device), labels.to(device)
            outputs = model_stgcn(data)
            loss = criterion(outputs, labels)
            test_loss += loss.detach()
            test_correct += (outputs.argmax(1) == labels).sum()
            test_total += len(labels)
            test_batches += 1
            eval_loop.set_postfix(
                loss=f"{loss.item():.4f}",
                acc=f"{100*test_correct.item()/test_total:.1f}%",
            )

    test_loss = test_loss.item() / test_batches
    test_acc = test_correct.item() / test_total * 100
    scheduler_stgcn.step()
    lr = scheduler_stgcn.get_last_lr()[0]

    history_stgcn["train_loss"].append(train_loss)
    history_stgcn["train_acc"].append(train_acc)
    history_stgcn["test_loss"].append(test_loss)
    history_stgcn["test_acc"].append(test_acc)
    history_stgcn["lr"].append(lr)

    print(f"{epoch:>6} {train_loss:>11.4f} {train_acc:>9.2f}% "
          f"{test_loss:>10.4f} {test_acc:>9.2f}% "
          f"{lr:>10.6f} {time.time()-t0:>5.1f}s")

    if test_acc > best_acc_stgcn:
        best_acc_stgcn = test_acc
        save_checkpoint(model_stgcn, optimizer_stgcn, epoch,
                        test_acc,
                        os.path.join(CKPT_DIR, "stgcn_final_best.pt"))
        print(f"        New best! Saved (acc={test_acc:.2f}%)")

    if epoch % 5 == 0:
        save_checkpoint(model_stgcn, optimizer_stgcn, epoch,
                        test_acc,
                        os.path.join(CKPT_DIR,
                                     f"stgcn_final_epoch{epoch}.pt"))
        print(f"         Periodic checkpoint (epoch {epoch})")

print(f"\n{'='*76}")
print(f"  ST-GCN Final Training Complete!")
print(f"  Seed               : {SEED}")
print(f"  Best test accuracy : {best_acc_stgcn:.2f}%")
print(f"{'='*76}")

Training ST-GCN for 60 epochs

 Epoch  Train Loss  Train Acc  Test Loss   Test Acc         LR   Time
────────────────────────────────────────────────────────────────────────────


Epoch  1/60 [Train]:   0%|                             | 0/1253 [00:00<?, ?it/s]

Epoch  1/60 [Eval] :   0%|                              | 0/516 [00:00<?, ?it/s]

     1      3.3312     16.03%     3.0602     23.81%   0.000994  63.2s
        New best! Saved (acc=23.81%)


Epoch  2/60 [Train]:   0%|                             | 0/1253 [00:00<?, ?it/s]

...

Epoch 59/60 [Eval] :   0%|                              | 0/516 [00:00<?, ?it/s]

    59      1.4405     79.82%     1.6020     73.20%   0.000016  62.7s
        New best! Saved (acc=73.20%)


Epoch 60/60 [Train]:   0%|                             | 0/1253 [00:00<?, ?it/s]

Epoch 60/60 [Eval] :   0%|                              | 0/516 [00:00<?, ?it/s]

    60      1.4350     80.24%     1.6096     72.78%   0.001000  62.5s
         Periodic checkpoint (epoch 60)

  ST-GCN Final Training Complete!
  Seed               : 42
  Best test accuracy : 73.20%


In [16]:
# Load best ST-GCN checkpoint
checkpoint = torch.load(
    os.path.join(CKPT_DIR, "stgcn_final_best.pt")
)
model_stgcn.load_state_dict(checkpoint["model_state"])
print(f"Loaded ST-GCN checkpoint")
print(f"  Epoch   : {checkpoint['epoch']}")
print(f"  Val acc : {checkpoint['val_acc']:.2f}%")

# Freeze all ST-GCN weights
for param in model_stgcn.parameters():
    param.requires_grad = False
model_stgcn.eval()

Loaded ST-GCN checkpoint
  Epoch   : 59
  Val acc : 73.20%


STGCN(
  (input_bn): BatchNorm1d(75, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (blocks): ModuleList(
    (0): STGCNBlock(
      (gcn): GraphConv(
        (conv): Conv2d(3, 64, kernel_size=(1, 1), stride=(1, 1))
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (tcn): Sequential(
        (0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (1): ReLU()
        (2): Conv2d(64, 64, kernel_size=(9, 1), stride=(1, 1), padding=(4, 0))
        (3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (4): Dropout(p=0.5, inplace=False)
      )
      (residual): Sequential(
        (0): Conv2d(3, 64, kernel_size=(1, 1), stride=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (relu): ReLU()
    )
    (1-2): 2 x STGCNBlock(
      (gcn): GraphConv(
        (conv): Conv2d(64, 64, 

Option 1 - MLP

In [17]:
class MLPStage2(nn.Module):
    """
    Final Stage 2 MLP with enriched skeleton features.

    Input features (630 total):
      ST-GCN logits   :  60  stage 1 class predictions
      Mean joint pose :  75  average position per joint
      Std joint pose  :  75  movement range per joint
      Max joint pose  :  75  peak joint positions
      Min joint pose  :  75  lowest joint positions
      Mean velocity   :  75  average frame-to-frame speed
      Std velocity    :  75  speed variance
      Mean bone vector:  60  average bone orientations (20 bones × 3)
      Std bone vector :  60  bone orientation variance

    Architecture: 630 - 512 - 512 - 256 - 256 - 60
    with BatchNorm, ReLU, Dropout and residual connections
    """
    def __init__(self, num_classes=60, dropout=0.4):
        super().__init__()

        # 60 logits + 450 joint stats + 120 bone stats = 630
        input_size = 60 + 450 + 120

        self.layer1 = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.layer2 = nn.Sequential(
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.down1 = nn.Linear(512, 256)
        self.layer3 = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.layer4 = nn.Sequential(
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(256, num_classes)

        # Bone connectivity - 20 pairs of connected joints
        self.bone_edges = [
            (0,1),(1,20),(20,2),(2,3),
            (20,4),(4,5),(5,6),(6,7),
            (20,8),(8,9),(9,10),(10,11),
            (0,12),(12,13),(13,14),(14,15),
            (0,16),(16,17),(17,18),(18,19),
        ]

    def extract_features(self, skeleton):
        """
        Extract joint statistics and bone features.
        skeleton : (batch, 3, 100, 25)
        returns  : (batch, 570)
        """
        B = skeleton.shape[0]

        # Joint position statistics across 100 frames
        mean_pose = skeleton.mean(dim=2).reshape(B, -1)
        std_pose  = skeleton.std(dim=2).reshape(B, -1)
        max_pose  = skeleton.max(dim=2).values.reshape(B, -1)
        min_pose  = skeleton.min(dim=2).values.reshape(B, -1)

        # Velocity statistics
        velocity  = skeleton[:, :, 1:, :] - skeleton[:, :, :-1, :]
        mean_vel  = velocity.mean(dim=2).reshape(B, -1)
        std_vel   = velocity.std(dim=2).reshape(B, -1)

        # Bone vector statistics
        # each bone = vector from joint i to joint j
        # captures limb orientations and segment angles
        bone_list = []
        for i, j in self.bone_edges:
            # difference vector for this bone across all frames
            bone = skeleton[:, :, :, i] - skeleton[:, :, :, j]
            bone_list.append(bone.mean(dim=2))
            bone_list.append(bone.std(dim=2))

        # stack all bone features (B, 20*2*3) = (B, 120)
        bone_features = torch.cat(bone_list, dim=1)

        return torch.cat([
            mean_pose, std_pose,
            max_pose,  min_pose,
            mean_vel,  std_vel,
            bone_features,
        ], dim=1)

    def forward(self, skeleton, stage1_logits):
        B = skeleton.shape[0]

        # Extract all features from skeleton
        skel_features = self.extract_features(skeleton)

        # Concatenate with ST-GCN predictions
        x = torch.cat([stage1_logits, skel_features], dim=1)

        # Forward through network with residual connections
        x   = self.layer1(x)         # (B, 512)
        x   = self.layer2(x) + x     # (B, 512) - residual
        res = self.down1(x)          # (B, 256)
        x   = self.layer3(x)         # (B, 256)
        x   = self.layer4(x) + res   # (B, 256) - residual
        return self.classifier(x)    # (B, 60)


# Instantiate
model_mlp = MLPStage2().to(device)
total = sum(p.numel() for p in model_mlp.parameters()
            if p.requires_grad)

# Test forward pass
with torch.no_grad():
    dummy_skel   = torch.randn(4, 3, 100, 25).to(device)
    dummy_logits = torch.randn(4, 60).to(device)
    out          = model_mlp(dummy_skel, dummy_logits)
    print(f"\nForward pass:")
    print(f"  Skeleton : {dummy_skel.shape}")
    print(f"  Logits   : {dummy_logits.shape}")
    print(f"  Output   : {out.shape} ")
print(f"MLP Stage 2 ready")


Forward pass:
  Skeleton : torch.Size([4, 3, 100, 25])
  Logits   : torch.Size([4, 60])
  Output   : torch.Size([4, 60]) 
MLP Stage 2 ready


In [19]:
criterion_s2  = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer_mlp = torch.optim.Adam(
    model_mlp.parameters(),
    lr = 1e-3,
    weight_decay = 1e-4,
)
NUM_EPOCHS_MLP = 60

scheduler_mlp = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_mlp,
    T_0     = 30,
    T_mult  = 1,
    eta_min = 1e-5,
)

best_acc_mlp = 0.0
history_mlp  = {
    "train_loss": [], "train_acc": [],
    "test_loss":  [], "test_acc":  [],
}

print(f"Training Improved MLP Stage 2 for {NUM_EPOCHS_MLP} epochs")
print(f"ST-GCN Stage 1 frozen at {checkpoint['val_acc']:.2f}%")
print(f"\n{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} "
      f"{'Test Loss':>10} {'Test Acc':>10} {'LR':>10} {'Time':>6}")
print("─" * 76)

for epoch in range(1, NUM_EPOCHS_MLP + 1):
    t0 = time.time()

    # Training
    model_mlp.train()
    model_stgcn.eval()

    train_loss = torch.tensor(0.0, device=device)
    train_correct = torch.tensor(0,   device=device)
    train_total = 0
    train_batches = 0

    loop = tqdm(train_loader,
                desc=f"Epoch {epoch:>2}/{NUM_EPOCHS_MLP} [Train]",
                leave=False, ncols=80)

    for data, labels in loop:
        data, labels = data.to(device), labels.to(device)

        # Frozen ST-GCN forward pass
        with torch.no_grad():
            stage1_logits = model_stgcn(data)

        # MLP forward + backward
        optimizer_mlp.zero_grad()
        outputs = model_mlp(data, stage1_logits)
        loss = criterion_s2(outputs, labels)
        loss.backward()
        optimizer_mlp.step()

        train_loss += loss.detach()
        train_correct += (outputs.detach().argmax(1) == labels).sum()
        train_total += len(labels)
        train_batches += 1

        loop.set_postfix(
            loss=f"{loss.item():.4f}",
            acc=f"{100*train_correct.item()/train_total:.1f}%",
        )

    train_loss = train_loss.item() / train_batches
    train_acc = train_correct.item() / train_total * 100

    # Evaluation
    model_mlp.eval()
    test_loss = torch.tensor(0.0, device=device)
    test_correct = torch.tensor(0, device=device)
    test_total = 0
    test_batches = 0

    eval_loop = tqdm(test_loader,
                     desc=f"Epoch {epoch:>2}/{NUM_EPOCHS_MLP} [Eval] ",
                     leave=False, ncols=80)

    with torch.no_grad():
        for data, labels in eval_loop:
            data, labels = data.to(device), labels.to(device)

            # Full pipeline
            stage1_logits = model_stgcn(data)
            outputs = model_mlp(data, stage1_logits)
            loss = criterion_s2(outputs, labels)

            test_loss += loss.detach()
            test_correct += (outputs.argmax(1) == labels).sum()
            test_total += len(labels)
            test_batches += 1

            eval_loop.set_postfix(
                loss=f"{loss.item():.4f}",
                acc=f"{100*test_correct.item()/test_total:.1f}%",
            )

    test_loss = test_loss.item() / test_batches
    test_acc  = test_correct.item() / test_total * 100
    scheduler_mlp.step()
    lr = scheduler_mlp.get_last_lr()[0]

    history_mlp["train_loss"].append(train_loss)
    history_mlp["train_acc"].append(train_acc)
    history_mlp["test_loss"].append(test_loss)
    history_mlp["test_acc"].append(test_acc)

    print(f"{epoch:>6} {train_loss:>11.4f} {train_acc:>9.2f}% "
          f"{test_loss:>10.4f} {test_acc:>9.2f}% "
          f"{lr:>10.6f} {time.time()-t0:>5.1f}s")

    if test_acc > best_acc_mlp:
        best_acc_mlp = test_acc
        save_checkpoint(model_mlp, optimizer_mlp, epoch,
                        test_acc,
                        os.path.join(CKPT_DIR, "mlp_stage2_best.pt"))
        print(f"         New best! Saved (acc={test_acc:.2f}%)")

    if epoch % 5 == 0:
        save_checkpoint(model_mlp, optimizer_mlp, epoch,
                        test_acc,
                        os.path.join(CKPT_DIR,
                                     f"mlp_stage2_epoch{epoch}.pt"))
        print(f"         Periodic checkpoint (epoch {epoch})")

print(f"\n{'='*76}")
print(f"  Pipeline Results")
print(f"  ────────────────────────────────────────────")
print(f"  ST-GCN standalone      : {checkpoint['val_acc']:.2f}%")
print(f"  ST-GCN - MLP pipeline  : {best_acc_mlp:.2f}%")
print(f"  ────────────────────────────────────────────")
print(f"{'='*76}")

Training Improved MLP Stage 2 for 60 epochs
ST-GCN Stage 1 frozen at 73.20%

 Epoch  Train Loss  Train Acc  Test Loss   Test Acc         LR   Time
────────────────────────────────────────────────────────────────────────────


Epoch  1/60 [Train]:   0%|                             | 0/1253 [00:00<?, ?it/s]

Epoch  1/60 [Eval] :   0%|                              | 0/516 [00:00<?, ?it/s]

     1      1.5731     74.32%     1.5338     74.60%   0.000997  26.4s
         New best! Saved (acc=74.60%)


Epoch  2/60 [Train]:   0%|                             | 0/1253 [00:00<?, ?it/s]

...

Epoch 59/60 [Eval] :   0%|                              | 0/516 [00:00<?, ?it/s]

    59      1.1474     86.12%     1.3858     77.87%   0.000013  26.5s


Epoch 60/60 [Train]:   0%|                             | 0/1253 [00:00<?, ?it/s]

Epoch 60/60 [Eval] :   0%|                              | 0/516 [00:00<?, ?it/s]

    60      1.1453     86.14%     1.3978     77.81%   0.001000  26.4s
         Periodic checkpoint (epoch 60)

  Pipeline Results
  ────────────────────────────────────────────
  ST-GCN standalone      : 73.20%
  ST-GCN - MLP pipeline  : 77.93%
  ────────────────────────────────────────────


Option 2 - LTSM

In [20]:
class LSTMStage2(nn.Module):
    """
    Stage 2 LTSM

    Inputs:
        skeleton : (batch, 3, 100, 25) raw skeleton sequence
        stage1_logits : (batch, 60) ST-GCN predictions

    Architecture:
        skeleton -> reshape ->  (batch, 100, 75)
                 ->  LSTM 2 layers, 256 hidden
                 ->  last hidden state (batch, 256)
                 ->  add ST-GCN logits (batch, 316)
                 ->  Linear -> 60 classes
    """
    def __init__(self,
                 input_size = 75,    # 25 joints × 3 coords per frame
                 hidden_size = 256,
                 num_layers  = 2,
                 num_classes = 60,
                 dropout = 0.5):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout = dropout,
        )

        self.dropout = nn.Dropout(dropout)

        # Late fusion: 256 LSTM features + 60 ST-GCN logits ->  60 classes
        self.classifier = nn.Linear(hidden_size + num_classes,
                                    num_classes)

    def forward(self, skeleton, stage1_logits):
        B = skeleton.shape[0]

        # Reshape skeleton for LSTM input
        # (batch, 3, 100, 25) ->  (batch, 100, 75)
        x = skeleton.permute(0, 2, 3, 1)
        x = x.reshape(B, 100, -1)

        # LSTM processes full sequence - captures temporal dynamics
        out, _ = self.lstm(x)
        out    = out[:, -1, :]              # last hidden state (B, 256)
        out    = self.dropout(out)

        # Late fusion with ST-GCN predictions
        out = torch.cat([out, stage1_logits], dim=1)  # (B, 316)

        return self.classifier(out)         # (B, 60)


# Instantiate
model_lstm_s2 = LSTMStage2().to(device)
total = sum(p.numel() for p in model_lstm_s2.parameters()
            if p.requires_grad)
print(f"Model         : LSTM Stage 2")
print(f"Parameters    : {total:,}")
print(f"LSTM input    : 75 (25 joints × 3 coords per frame)")
print(f"LSTM hidden   : 256 × 2 layers")
print(f"Fusion input  : 256 (LSTM) + 60 (ST-GCN) = 316")
print(f"Output        : 60 classes")

# Test forward pass
with torch.no_grad():
    dummy_skel   = torch.randn(4, 3, 100, 25).to(device)
    dummy_logits = torch.randn(4, 60).to(device)
    out          = model_lstm_s2(dummy_skel, dummy_logits)
    print(f"\nForward pass:")
    print(f"  Skeleton : {dummy_skel.shape}")
    print(f"  Logits   : {dummy_logits.shape}")
    print(f"  Output   : {out.shape}  : should be (4, 60)")
print(f"LSTM Stage 2 ready")

Model         : LSTM Stage 2
Parameters    : 886,348
LSTM input    : 75 (25 joints × 3 coords per frame)
LSTM hidden   : 256 × 2 layers
Fusion input  : 256 (LSTM) + 60 (ST-GCN) = 316
Output        : 60 classes

Forward pass:
  Skeleton : torch.Size([4, 3, 100, 25])
  Logits   : torch.Size([4, 60])
  Output   : torch.Size([4, 60])  : should be (4, 60)
LSTM Stage 2 ready


In [21]:
criterion_s2   = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer_lstm = torch.optim.Adam(
    model_lstm_s2.parameters(),
    lr           = 1e-3,
    weight_decay = 1e-4,
)

# Cosine annealing with warm restarts
scheduler_lstm = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_lstm,
    T_0     = 20,
    T_mult  = 1,
    eta_min = 1e-5,
)

best_acc_lstm_s2 = 0.0
history_lstm_s2  = {
    "train_loss": [], "train_acc": [],
    "test_loss":  [], "test_acc":  [],
}

NUM_EPOCHS_LSTM = 60

print(f"Training LSTM Stage 2 for {NUM_EPOCHS_LSTM} epochs")
print(f"ST-GCN Stage 1 frozen at {checkpoint['val_acc']:.2f}%")
print(f"\n{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} "
      f"{'Test Loss':>10} {'Test Acc':>10} {'LR':>10} {'Time':>6}")
print("─" * 76)

for epoch in range(1, NUM_EPOCHS_LSTM + 1):
    t0 = time.time()

    # Training
    model_lstm_s2.train()
    model_stgcn.eval()

    train_loss    = torch.tensor(0.0, device=device)
    train_correct = torch.tensor(0,   device=device)
    train_total   = 0
    train_batches = 0

    loop = tqdm(train_loader,
                desc=f"Epoch {epoch:>2}/{NUM_EPOCHS_LSTM} [Train]",
                leave=False, ncols=80)

    for data, labels in loop:
        data, labels = data.to(device), labels.to(device)

        # Stage 1 - frozen ST-GCN
        with torch.no_grad():
            stage1_logits = model_stgcn(data)

        # Stage 2 - LSTM
        optimizer_lstm.zero_grad()
        outputs = model_lstm_s2(data, stage1_logits)
        loss    = criterion_s2(outputs, labels)
        loss.backward()

        # Gradient clipping - essential for LSTM stability
        torch.nn.utils.clip_grad_norm_(
            model_lstm_s2.parameters(), max_norm=1.0
        )

        optimizer_lstm.step()

        train_loss    += loss.detach()
        train_correct += (outputs.detach().argmax(1) == labels).sum()
        train_total   += len(labels)
        train_batches += 1

        loop.set_postfix(
            loss=f"{loss.item():.4f}",
            acc=f"{100*train_correct.item()/train_total:.1f}%",
        )

    train_loss = train_loss.item() / train_batches
    train_acc  = train_correct.item() / train_total * 100

    # Evaluation
    model_lstm_s2.eval()
    test_loss    = torch.tensor(0.0, device=device)
    test_correct = torch.tensor(0,   device=device)
    test_total   = 0
    test_batches = 0

    eval_loop = tqdm(test_loader,
                     desc=f"Epoch {epoch:>2}/{NUM_EPOCHS_LSTM} [Eval] ",
                     leave=False, ncols=80)

    with torch.no_grad():
        for data, labels in eval_loop:
            data, labels = data.to(device), labels.to(device)
            stage1_logits = model_stgcn(data)
            outputs = model_lstm_s2(data, stage1_logits)
            loss          = criterion_s2(outputs, labels)
            test_loss    += loss.detach()
            test_correct += (outputs.argmax(1) == labels).sum()
            test_total   += len(labels)
            test_batches += 1
            eval_loop.set_postfix(
                loss=f"{loss.item():.4f}",
                acc=f"{100*test_correct.item()/test_total:.1f}%",
            )

    test_loss = test_loss.item() / test_batches
    test_acc  = test_correct.item() / test_total * 100
    scheduler_lstm.step()
    lr = scheduler_lstm.get_last_lr()[0]

    history_lstm_s2["train_loss"].append(train_loss)
    history_lstm_s2["train_acc"].append(train_acc)
    history_lstm_s2["test_loss"].append(test_loss)
    history_lstm_s2["test_acc"].append(test_acc)

    print(f"{epoch:>6} {train_loss:>11.4f} {train_acc:>9.2f}% "
          f"{test_loss:>10.4f} {test_acc:>9.2f}% "
          f"{lr:>10.6f} {time.time()-t0:>5.1f}s")

    if test_acc > best_acc_lstm_s2:
        best_acc_lstm_s2 = test_acc
        save_checkpoint(model_lstm_s2, optimizer_lstm, epoch,
                        test_acc,
                        os.path.join(CKPT_DIR, "lstm_stage2_best.pt"))
        print(f"        New best! Saved (acc={test_acc:.2f}%)")

    if epoch % 10 == 0:
        save_checkpoint(model_lstm_s2, optimizer_lstm, epoch,
                        test_acc,
                        os.path.join(CKPT_DIR,
                                     f"lstm_stage2_epoch{epoch}.pt"))
        print(f"        Periodic checkpoint (epoch {epoch})")

print(f"\n{'='*76}")
print(f"  ────────────────────────────────────────────────────")
print(f"  ST-GCN standalone        : {checkpoint['val_acc']:.2f}%")
print(f"  ST-GCN -> LSTM pipeline   : {best_acc_lstm_s2:.2f}%")
print(f"  ────────────────────────────────────────────────────")
print(f"{'='*76}")

Training LSTM Stage 2 for 60 epochs
ST-GCN Stage 1 frozen at 73.20%

 Epoch  Train Loss  Train Acc  Test Loss   Test Acc         LR   Time
────────────────────────────────────────────────────────────────────────────


Epoch  1/60 [Train]:   0%|                             | 0/1253 [00:00<?, ?it/s]

Epoch  1/60 [Eval] :   0%|                              | 0/516 [00:00<?, ?it/s]

     1      1.7812     71.32%     1.5725     74.42%   0.000994  32.6s
        New best! Saved (acc=74.42%)


Epoch  2/60 [Train]:   0%|                             | 0/1253 [00:00<?, ?it/s]

...

Epoch 59/60 [Eval] :   0%|                              | 0/516 [00:00<?, ?it/s]

    59      1.2901     84.50%     1.5050     76.87%   0.000016  32.5s


Epoch 60/60 [Train]:   0%|                             | 0/1253 [00:00<?, ?it/s]

Epoch 60/60 [Eval] :   0%|                              | 0/516 [00:00<?, ?it/s]

    60      1.2906     84.62%     1.5041     76.84%   0.001000  33.3s
        Periodic checkpoint (epoch 60)

  ────────────────────────────────────────────────────
  ST-GCN standalone        : 73.20%
  ST-GCN -> LSTM pipeline   : 76.95%
  ────────────────────────────────────────────────────
